# Naive Bayes & NLP - Assignment


In [2]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

from sklearn.utils import shuffle

from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import GaussianNB

import nltk
from nltk.corpus import stopwords
import re #regular expressions
from bs4 import BeautifulSoup
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

## PART 1 - Sentiment analysis 

Classify the sentiment of text messages as being positive or negative.
The training data and test data can be found in:  *sentiment_train.csv* and *sentiment_test.csv*.

Follow these steps:

1. Analys of the data.
2. Preprocessing of the text.
3. Convert to bag-of-words.
4. Training the classifiers.
5. Testing and hyperparameter tuning.

In [3]:
# Import the datasets
dataset_train = pd.read_csv('sentiment_train.csv')
dataset_test = pd.read_csv('sentiment_test.csv')

dataset_train.head(10)

,sentiment,text
0,1,The Da Vinci Code book is just awesome.
1,0,"Oh, and Brokeback Mountain was a terrible movie."
2,1,"He's like,'YEAH I GOT ACNE AND I LOVE BROKEBAC..."
3,1,1st and 2nd Harry Potter movies are clearly th...
4,0,Mission Impossible 3 was quite boring.
5,0,So Brokeback Mountain was really depressing.
6,1,I love the Harry Potter series if you can coun...
7,0,I thought Brokeback Mountain was an awful movie.
8,1,Harry Potter is AWESOME I don't care if anyone...
9,1,dudeee i LOVED brokeback mountain!!!!


### 1. Analysis of the data

- Look for missing values.
- Is the dataset balanced?
- Is there a correlation between the length of a text and the sentiment?

In [4]:
# analysis of the data

# Looking for missing values
print(dataset_train.isnull().sum())

# Check if the dataset is balanced
print("\n", dataset_train['sentiment'].value_counts(normalize=True))

# Check if there is a correlation between the length of a text and the sentiment
corr = dataset_train['text'].apply(len).corr(dataset_train['sentiment'])
print("\nCorrelation between length and sentiment:", corr)


sentiment    0
text         0
dtype: int64

 sentiment
1    0.571646
0    0.428354
Name: proportion, dtype: float64

Correlation between length and sentiment: -0.04663722044425933


### Result of analysis
There is no missing values in the train dataset.

The dataset is balanced and the result shows that the dataset consists of 57% class 1 (sentiment = 1) and 43% class 0 (sentiment = 0).

The correlation shows that the lenth of the text has a negative impact on the sentiment value. If the length increase by 1, the sentiment decrease by 0.046. Which indicates that longer texts has a lower chance of getting sentiment 1.

### 2. Preprocessing of the text

- clean the text: remove stopwords, non-literal characters, ... 
- Apply stemming.

In [5]:
# Split into feature and target
X_train = dataset_train.text.values
y_train = dataset_train.sentiment.values

X_test = dataset_test.text.values
y_test = dataset_test.sentiment.values

In [6]:
# Preprocessing of the text
nltk.download('stopwords')

def text_preprocessing(text, language, minWordSize):

    # Remove html
    text_no_html = BeautifulSoup(str(text),"html.parser").get_text()

    # Remove non-letters
    text_alpha_chars = re.sub("[^a-zA-Z]", " ", str(text_no_html))

    # Convert to lower-case
    text_lower = text_alpha_chars.lower()

    # Remove stop words
    stops = set(stopwords.words(language))
    text_no_stop_words = ' '

    for w in text_lower.split():
        if w not in stops:
            text_no_stop_words = text_no_stop_words + w + ' '
    
    # Apply stemming
    text_stemmer = ' '
    stemmer = SnowballStemmer(language)
    for w in text_no_stop_words.split():
        text_stemmer = text_stemmer + stemmer.stem(w) + ' '
    
    # Remove short words
    text_no_short_words = ' '
    for w in text_stemmer.split():
        if len(w) >= minWordSize:
            text_no_short_words = text_no_short_words + w + ' '
    
    return text_no_short_words

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\carll\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### 3. Convert to bag-of-words

Use the CountVectorizer and TfidfTransformer to create a bag-of-words model. 

More info: 

https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html 

https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html#sklearn.feature_extraction.text.TfidfTransformer 

https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html#sklearn.feature_extraction.text.TfidfVectorizer



In [7]:
# Convert to bag-of-words
language = 'english'
minWordLength = 2

for i in range(X_train.size):
    X_train[i] = text_preprocessing(X_train[i], language, minWordLength)

for i in range(X_test.size):
    X_test[i] = text_preprocessing(X_test[i], language, minWordLength)

count_vect = CountVectorizer()
X_train_bag_of_words = count_vect.fit_transform(X_train)
X_test_bag_of_words = count_vect.transform(X_test)

print(X_train_bag_of_words)

tfidf_transformer = TfidfTransformer()
tf_transformer = TfidfTransformer(use_idf=True).fit(X_train_bag_of_words)
X_train_tf = tf_transformer.transform(X_train_bag_of_words)
X_test_tf = tf_transformer.transform(X_test_bag_of_words)

print("\n", X_train_bag_of_words.shape)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 38303 stored elements and shape (5918, 1573)>
  Coords	Values
  (0, 310)	1
  (0, 1492)	1
  (0, 243)	1
  (0, 147)	1
  (0, 86)	1
  (1, 979)	1
  (1, 164)	1
  (1, 931)	1
  (1, 1379)	1
  (1, 934)	1
  (2, 164)	1
  (2, 931)	1
  (2, 827)	1
  (2, 1563)	1
  (2, 592)	1
  (2, 12)	1
  (2, 852)	1
  (3, 934)	1
  (3, 1315)	1
  (3, 952)	1
  (3, 627)	1
  (3, 1072)	1
  (3, 230)	1
  (3, 118)	1
  (3, 550)	1
  :	:
  (5914, 1360)	1
  (5914, 595)	1
  (5914, 780)	1
  (5914, 321)	1
  (5914, 534)	1
  (5915, 164)	1
  (5915, 931)	1
  (5915, 852)	1
  (5915, 393)	1
  (5916, 310)	1
  (5916, 1492)	1
  (5916, 243)	1
  (5916, 852)	1
  (5916, 1027)	1
  (5916, 940)	1
  (5916, 1547)	1
  (5916, 791)	1
  (5917, 627)	1
  (5917, 1072)	1
  (5917, 630)	1
  (5917, 605)	1
  (5917, 791)	1
  (5917, 1503)	1
  (5917, 41)	1
  (5917, 294)	1

 (5918, 1573)


### 4. Train the classifiers

Train 3 different types of classifiers: naive bayes classifier, logistic regression classifier and a Support Vector Machine classifier.



In [8]:
# Train naive bayes
NBclassifier = MultinomialNB(alpha=1)

NBclassifier.fit(X_train_tf, y_train)

y_pred = NBclassifier.predict(X_test_tf)
print("Classification Report:\n", classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nAccuracy Score:", accuracy_score(y_test, y_pred))

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.96      0.97       440
           1       0.97      0.99      0.98       560

    accuracy                           0.98      1000
   macro avg       0.98      0.97      0.98      1000
weighted avg       0.98      0.98      0.98      1000


Confusion Matrix:
 [[421  19]
 [  4 556]]

Accuracy Score: 0.977


In [13]:
# Train logistic regression
log_reg = LogisticRegression(C=1, max_iter=1000)

log_reg.fit(X_train_tf, y_train)

y_pred = log_reg.predict(X_test_tf)

print("Classification report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nAccuracy Score:", accuracy_score(y_test, y_pred))

Classification report:
               precision    recall  f1-score   support

           0       1.00      0.99      0.99       440
           1       0.99      1.00      0.99       560

    accuracy                           0.99      1000
   macro avg       0.99      0.99      0.99      1000
weighted avg       0.99      0.99      0.99      1000


Confusion Matrix:
 [[434   6]
 [  2 558]]

Accuracy Score: 0.992


In [12]:
# Train SVM
svm_model = svm.SVC(C=1, kernel='linear')

svm_model.fit(X_train_tf, y_train)

y_pred = svm_model.predict(X_test_tf)

print("Classification report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nAccuracy Score:", accuracy_score(y_test, y_pred))

Classification report:
               precision    recall  f1-score   support

           0       1.00      0.99      0.99       440
           1       0.99      1.00      1.00       560

    accuracy                           0.99      1000
   macro avg       1.00      0.99      0.99      1000
weighted avg       1.00      0.99      0.99      1000


Confusion Matrix:
 [[436   4]
 [  1 559]]

Accuracy Score: 0.995


### 5. Testing and hyperparameter tuning

- Use grid-search, random search or Bayes Optimization for hyperparameter tuning.
- Which classifier has the best performance? bring arguments in terms of f1-score and computational time.

In [23]:
# Testing and hyperparameter tuning (Logistic Regression)
from sklearn.model_selection import GridSearchCV
import time

start = time.time()

log_param_grid = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "penalty": ['l1', 'l2'],
    "solver": ['liblinear']
}

log_grid = GridSearchCV(
    estimator=log_reg,
    param_grid=log_param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

log_grid.fit(X_train_tf, y_train)
print(f"Logistic Regression training time: {time.time() - start:.2f}s")

y_pred_log = log_grid.predict(X_test_tf)
print("\nAccuracy:", accuracy_score(y_test, y_pred_log))
print("\nClassification Report:\n", classification_report(y_test, y_pred_log))
print("\nBest Parameters (log):", log_grid.best_params_)
print("Best Score (log):", log_grid.best_score_)

Logistic Regression training time: 0.17s

Accuracy: 0.997

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.99      1.00       440
           1       0.99      1.00      1.00       560

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000


Best Parameters (log): {'C': 100, 'penalty': 'l1', 'solver': 'liblinear'}
Best Score (log): 0.9962819275319275


In [22]:
# Testing and hyperparameter tuning (Naive Bayes)
start = time.time()

nb_param_grid = {
    "alpha": [0.001, 0.01, 0.1, 0.5, 1, 2, 5, 10],
    "fit_prior": [True, False]
}

nb_grid = GridSearchCV(
    estimator=NBclassifier,
    param_grid=nb_param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

nb_grid.fit(X_train_tf, y_train)
print(f"Naive Bayes training time: {time.time() - start:.2f}s")

y_pred_nb = nb_grid.predict(X_test_tf)
print("\nAccuracy:", accuracy_score(y_test, y_pred_nb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_nb))
print("\nBest Parameters (NB):", nb_grid.best_params_)
print("Best Score (NB):", nb_grid.best_score_)

Naive Bayes training time: 0.09s

Accuracy: 0.977

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.96      0.97       440
           1       0.97      0.99      0.98       560

    accuracy                           0.98      1000
   macro avg       0.98      0.97      0.98      1000
weighted avg       0.98      0.98      0.98      1000


Best Parameters (NB): {'alpha': 1, 'fit_prior': True}
Best Score (NB): 0.9836095816865049


In [21]:
# Testing and hyperparameter tuning (Support Vector Machine)
start = time.time()

svm_param_grid = {
    "C": [0.1, 1, 10, 100],
    "kernel": ['linear', 'rbf'],
    "gamma": ['scale', 'auto'],
}

svm_grid = GridSearchCV(
    estimator=svm.SVC(),
    param_grid=svm_param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

svm_grid.fit(X_train_tf, y_train)
print(f"SVM training time: {time.time() - start:.2f}s")

y_pred_svm = svm_grid.predict(X_test_tf)
print("\nAccuracy:", accuracy_score(y_test, y_pred_svm))
print("\nClassification Report:\n", classification_report(y_test, y_pred_svm))
print("\nBest Parameters (SVM):", svm_grid.best_params_)
print("Best Score (SVM):", svm_grid.best_score_)

SVM training time: 4.08s

Accuracy: 0.995

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.99      0.99       440
           1       0.99      1.00      1.00       560

    accuracy                           0.99      1000
   macro avg       1.00      0.99      0.99      1000
weighted avg       1.00      0.99      0.99      1000


Best Parameters (SVM): {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Best Score (SVM): 0.9966201937355784


### Conflusions after hyperparameter tuning
After compelting the hyperparameter tuning of the three different models (Logistic Regression, Naive Bayes, Support Vector Machine) the result shows that the **``LogisticRegression``** model has the best performance.

By using hyperparameter tuning the model reached a accuracy of 99.6% with an F1-score of 1.00 for both class 0 and class 1.

The computational time for the three different model varied a bit and **``Naive Bayes``** model was the fastest model followed by the **``Logistic Regression``** model and lastely the **``Support Vector Machine``** model. 

## PART 2  - Sarcasm detection

Train a classifier to detect sarcasm in newspaper headlines.

Use the 'Sarcasm.json' dataset.

Train, evaluate and compare naive bayes, logistic regression and support vector machines

In [4]:
dataset = pd.read_json('Sarcasm.json')
dataset.head()

,headline,is_sarcastic
0,former versace store clerk sues over secret 'b...,0
1,the 'roseanne' revival catches up to our thorn...,0
10,airline passengers tackle man who rushes cockp...,0
100,demi lovato drops emotional 'nightingale' musi...,0
1000,california marijuana businesses get their firs...,0
